In [ ]:
import requests
import time
import base64
from PIL import Image

In [ ]:
#sanity check with /v1/models
# base url example: base_url = "http://granite-vision-model-predictor.user9.svc.cluster.local"
# enter the inference endpoints/url from the deployed model

base_url = "<enter your base url here>"
requests.get(f"{base_url}/v1/models").json()

In [ ]:
#Perform a basic text-only test

base_url = "http://granite-vision-model-predictor.user9.svc.cluster.local"

resp = requests.post(
    f"{base_url}/v1/chat/completions",
    json={
        "model": "granite-vision-model",
        "messages": [{"role": "user", "content": "Hello, are you working?"}],
        "max_tokens": 100
    }
)
print(resp.status_code)
print(resp.json())

In [ ]:
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_b64 = encode_image("chart.png")
print("Encoded length:", len(image_b64))  # sanity check — should be a long string, not empty

In [ ]:
#Perform a multimodal test - send an image + question

# Check image size first
img = Image.open("chart.png")
print("Image size:", img.size)

#For a 619x344 image on CPU-only inference the range should 30-90 seconds for inference response.
start = time.time()
resp = requests.post(
    f"{base_url}/v1/chat/completions",
    json={
        "model": "granite-vision-model",
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": "What does this chart show?"},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}
        ],
        "max_tokens": 100
    },
    timeout=120
)
print("Elapsed:", time.time() - start, "seconds")
print(resp.status_code)
print(resp.json())


<b>Let's talk Response Time </b>
The response quality (60-90s) is genuinely good — it correctly identified the image as an architecture diagram, described the high-availability/replication concept, and mentioned load balancers and data center replication — that's real diagram comprehension, not a generic answer.